# LLM Fiscal and Monetary Policy Classification Demo

This notebook demonstrates the usage of all fiscal and monetary policy classification prompts:
- **Monetary Stance Classification** (4 variants)
- **Monetary Agreement Classification** (4 variants) 
- **Fiscal Stance Classification** (4 variants)
- **Fiscal Agreement Classification** (4 variants)

Each category has: simple, with_definitions, chain_of_thought, and few_shot versions.

In [4]:
from dotenv import load_dotenv
load_dotenv('../../.env') ## load api key from .env file
import os,sys
sys.path.insert(0, '../../libs')
import pydantic
from pydantic import BaseModel, Field
from typing import List, Literal
from llm_factory_openai import LLMAgent
from prompt_utils import load_prompt, format_messages
import glob
import json

## Setup LLM Agent

In [5]:
llm_agent = LLMAgent(api_key=os.getenv('OPENAI_API_KEY'),
               model='gpt-5-nano',  # Using GPT-4o for better performance
               temperature=1)  # Lower temperature for more consistent results

# test connection
print(llm_agent.test_connection())

Hello! Yes, I can respond. How can I help you today? I can answer questions, explain concepts, draft emails or messages, help with writing or coding, solve math problems, brainstorm ideas, translate text, plan activities, and more. Tell me what you’d like to do or ask a question, and I’ll dive in.


## Load All Prompt Templates

In [6]:
# Load all prompt files
prompt_dir = "../../src/Traction/prompts/"
prompt_files = glob.glob(f"{prompt_dir}*.md")

# Organize prompts by category
prompts = {
    'monetary_stance': [],
    'monetary_agreement': [],
    'fiscal_stance': [],
    'fiscal_agreement': []
}

for file_path in prompt_files:
    file_name = os.path.basename(file_path)
    if file_name.startswith('monetary_stance_'):
        prompts['monetary_stance'].append(file_path)
    elif file_name.startswith('monetary_agreement_'):
        prompts['monetary_agreement'].append(file_path)
    elif file_name.startswith('fiscal_stance_'):
        prompts['fiscal_stance'].append(file_path)
    elif file_name.startswith('fiscal_agreement_'):
        prompts['fiscal_agreement'].append(file_path)

# Display loaded prompts
for category, files in prompts.items():
    print(f"\n{category.upper()}:")
    for file_path in sorted(files):
        print(f"  - {os.path.basename(file_path)}")


MONETARY_STANCE:
  - monetary_stance_chain_of_thought.md
  - monetary_stance_few_shot.md
  - monetary_stance_simple.md
  - monetary_stance_with_definitions.md

MONETARY_AGREEMENT:
  - monetary_agreement_chain_of_thought.md
  - monetary_agreement_few_shot.md
  - monetary_agreement_simple.md
  - monetary_agreement_with_definitions.md

FISCAL_STANCE:
  - fiscal_stance_chain_of_thought.md
  - fiscal_stance_few_shot.md
  - fiscal_stance_simple.md
  - fiscal_stance_with_definitions.md

FISCAL_AGREEMENT:
  - fiscal_agreement_chain_of_thought.md
  - fiscal_agreement_few_shot.md
  - fiscal_agreement_simple.md
  - fiscal_agreement_with_definitions.md


## Define Response Models (from schema.py)

In [7]:
# Import response models from schema.py
sys.path.insert(0, '../../src/Traction/prompts')
from schema import (
    MonetaryStanceResponse,
    MonetaryStanceChainOfThoughtResponse,
    MonetaryAgreementResponse,
    MonetaryAgreementChainOfThoughtResponse,
    FiscalStanceResponse,
    FiscalStanceChainOfThoughtResponse,
    FiscalAgreementResponse,
    FiscalAgreementChainOfThoughtResponse
)

print("Response models imported successfully from schema.py")

Response models imported successfully from schema.py


## Sample Data for Testing

In [8]:
# Sample monetary policy texts
monetary_samples = {
    'accommodative_tightening': {
        'text': "The accommodative monetary policy is appropriate given current economic conditions, but the central bank should gradually move towards a neutral stance in 2024. Pass-through from recent exchange rate changes to inflation should be closely monitored.",
        'country': 'Brazil',
        'year': '2024',
        'text_author': 'IMF staff'
    },
    'restrictive_loosening': {
        'text': "The current tight monetary policy stance has successfully anchored inflation expectations. However, with economic growth slowing and inflationary pressures subsiding, there may be room for modest policy easing in the coming quarters.",
        'country': 'Turkey',
        'year': '2024',
        'text_author': 'country authorities'
    }
}

# Sample monetary agreement texts
monetary_agreement_samples = {
    'disagreement': {
        'staff': "The accommodative monetary policy is appropriate, but should gradually move towards a neutral stance in 2024. Pass-through from the exchange rate and from recent reforms to inflation should be closely monitored.",
        'authority': "Monetary policy will remain broadly accommodative to support growth. The central bank maintains the policy rate and will continue to closely monitor domestic and international conditions before adjusting its policy stance.",
        'country': 'Colombia',
        'year': '2024'
    },
    'mostly_agree': {
        'staff': "The monetary policy stance is broadly appropriate given the low-inflation environment. Further excess liquidity absorption should proceed at a measured pace to avoid sharp increases in market interest rates.",
        'authority': "The central bank's monetary policy will remain cautiously accommodative to maintain price stability. Efforts to gradually reduce excess domestic liquidity will be enhanced to improve the monetary policy transmission mechanism.",
        'country': 'Mauritius',
        'year': '2024'
    }
}

# Sample fiscal stance texts  
fiscal_stance_samples = {
    'consolidation_vs_expansion': {
        'staff': "The fiscal deficit has increased significantly and consolidation measures are urgently needed. Staff recommends reducing government expenditure by 2% of GDP over the next two years through efficiency improvements.",
        'authority': "The government will continue with supportive fiscal policies to sustain economic recovery. Additional infrastructure spending and social programs are planned for the next fiscal year to boost growth and employment.",
        'country': 'South Africa',
        'year': '2024'
    },
    'gradual_adjustment': {
        'staff': "The current fiscal stance is appropriately supportive given the economic environment. As growth recovers, the authorities should gradually move toward fiscal consolidation to ensure debt sustainability.",
        'authority': "The authorities acknowledge the need for fiscal balance over the medium term. They plan to implement gradual expenditure adjustments while protecting essential social spending.",
        'country': 'Chile',
        'year': '2024'
    }
}

# Sample fiscal agreement texts
fiscal_agreement_samples = {
    'tax_policy_disagreement': {
        'staff': "Staff recommends implementing comprehensive tax reforms to broaden the tax base and improve revenue mobilization. The current tax-to-GDP ratio is below regional averages and needs improvement.",
        'authority': "The government has prioritized improving public expenditure efficiency rather than increasing tax burdens. Recent initiatives focus on better targeting of social programs and reducing waste.",
        'country': 'Peru',
        'year': '2024'
    },
    'debt_management_agreement': {
        'staff': "The authorities' debt management strategy is appropriate and should help maintain fiscal sustainability. Continued focus on lengthening debt maturity and reducing foreign currency exposure is welcomed.",
        'authority': "The government remains committed to prudent debt management practices. Our strategy focuses on optimizing the debt portfolio structure and maintaining market access at reasonable costs.",
        'country': 'Mexico',
        'year': '2024'
    }
}

print("Sample data loaded successfully!")
print(f"Monetary samples: {len(monetary_samples)}")
print(f"Monetary agreement samples: {len(monetary_agreement_samples)}")
print(f"Fiscal stance samples: {len(fiscal_stance_samples)}")
print(f"Fiscal agreement samples: {len(fiscal_agreement_samples)}")

Sample data loaded successfully!
Monetary samples: 2
Monetary agreement samples: 2
Fiscal stance samples: 2
Fiscal agreement samples: 2


## 1. Monetary Stance Classification Tests

In [9]:
def test_monetary_stance(prompt_file, sample_data, sample_name):
    """Test monetary stance classification with a given prompt file and sample data"""
    print(f"\n=== Testing {os.path.basename(prompt_file)} with {sample_name} ===")
    
    # Load prompt template
    prompt_template = load_prompt(prompt_file).sections
    
    # Format the template
    formatted_template = prompt_template.copy()
    formatted_template["user"] = formatted_template["user"].format(
        COUNTRY=sample_data['country'],
        YEAR=sample_data['year'],
        TEXT=sample_data['text']
    )
    
    # Replace {TEXT_AUTHOR} in system prompt if it exists
    if '{TEXT_AUTHOR}' in formatted_template["system"]:
        formatted_template["system"] = formatted_template["system"].replace(
            '{TEXT_AUTHOR}', sample_data['text_author']
        )
    
    messages = format_messages(formatted_template, add_schema=True)
    
    # Choose response format based on prompt type
    if 'chain_of_thought' in prompt_file:
        response_format = MonetaryStanceChainOfThoughtResponse
    else:
        response_format = MonetaryStanceResponse
    
    # Get response
    try:
        response = llm_agent.get_response_content(messages, reasoning_effort='low', response_format=response_format)
        print(f"Current stance: {response.stance_current}")
        print(f"Future direction: {response.stance_future}")
        if hasattr(response, 'reason'):
            print(f"Reasoning: {response.reason}")
    except Exception as e:
        print(f"Error: {e}")

# Test all monetary stance prompts with sample data
for prompt_file in sorted(prompts['monetary_stance']):
    for sample_name, sample_data in monetary_samples.items():
        test_monetary_stance(prompt_file, sample_data, sample_name)
        break  # Only test with first sample for brevity


=== Testing monetary_stance_chain_of_thought.md with accommodative_tightening ===
Current stance: accommodative
Future direction: tightening
Reasoning: The text explicitly states that monetary policy is accommodative but should gradually move towards a neutral stance in 2024, implying a shift away from accommodative toward neutral, i.e., a tightening relative to the current stance. Pass-through monitoring is about inflation dynamics, not a different policy stance.

=== Testing monetary_stance_few_shot.md with accommodative_tightening ===
Current stance: accommodative
Future direction: tightening

=== Testing monetary_stance_simple.md with accommodative_tightening ===
Current stance: accommodative
Future direction: tightening

=== Testing monetary_stance_with_definitions.md with accommodative_tightening ===
Current stance: accommodative
Future direction: tightening bias


## 2. Monetary Agreement Classification Tests

In [10]:
def test_monetary_agreement(prompt_file, sample_data, sample_name):
    """Test monetary agreement classification with a given prompt file and sample data"""
    print(f"\n=== Testing {os.path.basename(prompt_file)} with {sample_name} ===")
    
    # Load prompt template
    prompt_template = load_prompt(prompt_file).sections
    
    # Format the template
    formatted_template = prompt_template.copy()
    formatted_template["user"] = formatted_template["user"].format(
        COUNTRY=sample_data['country'],
        YEAR=sample_data['year'],
        STAFF_TEXT=sample_data['staff'],
        AUTHORITY_TEXT=sample_data['authority']
    )
    
    messages = format_messages(formatted_template, add_schema=True)
    
    # Choose response format based on prompt type
    if 'chain_of_thought' in prompt_file:
        response_format = MonetaryAgreementChainOfThoughtResponse
    else:
        response_format = MonetaryAgreementResponse
    
    # Get response
    try:
        response = llm_agent.get_response_content(messages, reasoning_effort='low', response_format=response_format)
        print(f"Agreement: {response.agreement}")
        print(f"Disagreement areas: '{response.disagreement_areas}'")
        if hasattr(response, 'reason'):
            print(f"Reasoning: {response.reason}")
    except Exception as e:
        print(f"Error: {e}")

# Test all monetary agreement prompts with sample data
for prompt_file in sorted(prompts['monetary_agreement']):
    for sample_name, sample_data in monetary_agreement_samples.items():
        test_monetary_agreement(prompt_file, sample_data, sample_name)
        break  # Only test with first sample for brevity


=== Testing monetary_agreement_chain_of_thought.md with disagreement ===
Agreement: disagreement exists
Disagreement areas: 'Future Policy Direction; Monetary Policy Stance'
Reasoning: IMF staff advocates gradual move towards a neutral stance in 2024 while monitoring pass-through; the authority states the policy will remain broadly accommodative with no rate change for now and to monitor conditions before adjusting stance. These reflect differing expectations about the stance and path of policy in the near term.

=== Testing monetary_agreement_few_shot.md with disagreement ===
Agreement: disagreement exists
Disagreement areas: 'Current policy stance; Future policy direction'

=== Testing monetary_agreement_simple.md with disagreement ===
Agreement: disagreement exists
Disagreement areas: 'Current Policy Stance; Future Policy Direction'

=== Testing monetary_agreement_with_definitions.md with disagreement ===
Agreement: disagreement exists
Disagreement areas: 'Future Policy Direction'


## 3. Fiscal Stance Classification Tests

In [11]:
def test_fiscal_stance(prompt_file, sample_data, sample_name):
    """Test fiscal stance classification with a given prompt file and sample data"""
    print(f"\n=== Testing {os.path.basename(prompt_file)} with {sample_name} ===")
    
    # Load prompt template
    prompt_template = load_prompt(prompt_file).sections
    
    # Format the template
    formatted_template = prompt_template.copy()
    formatted_template["user"] = formatted_template["user"].format(
        STAFF_TEXT=sample_data['staff'],
        AUTHORITY_TEXT=sample_data['authority']
    )
    
    messages = format_messages(formatted_template, add_schema=True)
    
    # Choose response format based on prompt type
    if 'chain_of_thought' in prompt_file:
        response_format = FiscalStanceChainOfThoughtResponse
    else:
        response_format = FiscalStanceResponse
    
    # Get response
    try:
        response = llm_agent.get_response_content(messages, reasoning_effort='low', response_format=response_format)
        print(f"Staff current: {response.staff_current}")
        print(f"Staff future: {response.staff_future}")
        print(f"Authority current: {response.authority_current}")
        print(f"Authority future: {response.authority_future}")
        print(f"Agreement other: {response.agreement_other}")
        if hasattr(response, 'reason'):
            print(f"Reasoning: {response.reason[:200]}...")  # Truncate for readability
    except Exception as e:
        print(f"Error: {e}")

# Test all fiscal stance prompts with sample data
for prompt_file in sorted(prompts['fiscal_stance']):
    for sample_name, sample_data in fiscal_stance_samples.items():
        test_fiscal_stance(prompt_file, sample_data, sample_name)
        break  # Only test with first sample for brevity


=== Testing fiscal_stance_chain_of_thought.md with consolidation_vs_expansion ===
Staff current: contractionary
Staff future: contractionary
Authority current: expansionary
Authority future: expansionary
Agreement other: irrelevant
Reasoning: Staff calls for expenditure reduction of 2% of GDP over two years, indicating contractionary stance both now and in the near future. The authority advocates ongoing supportive fiscal policies with upc...

=== Testing fiscal_stance_few_shot.md with consolidation_vs_expansion ===
Staff current: contractionary
Staff future: contractionary
Authority current: expansionary
Authority future: expansionary
Agreement other: disagree

=== Testing fiscal_stance_simple.md with consolidation_vs_expansion ===
Staff current: contractionary
Staff future: contractionary
Authority current: expansionary
Authority future: expansionary
Agreement other: irrelevant

=== Testing fiscal_stance_with_definitions.md with consolidation_vs_expansion ===
Staff current: moderate

## 4. Fiscal Agreement Classification Tests

In [12]:
def test_fiscal_agreement(prompt_file, sample_data, sample_name):
    """Test fiscal agreement classification with a given prompt file and sample data"""
    print(f"\n=== Testing {os.path.basename(prompt_file)} with {sample_name} ===")
    
    # Load prompt template
    prompt_template = load_prompt(prompt_file).sections
    
    # Format the template
    formatted_template = prompt_template.copy()
    formatted_template["user"] = formatted_template["user"].format(
        COUNTRY=sample_data['country'],
        YEAR=sample_data['year'],
        STAFF_TEXT=sample_data['staff'],
        AUTHORITY_TEXT=sample_data['authority']
    )
    
    messages = format_messages(formatted_template, add_schema=True)
    
    # Choose response format based on prompt type
    if 'chain_of_thought' in prompt_file:
        response_format = FiscalAgreementChainOfThoughtResponse
    else:
        response_format = FiscalAgreementResponse
    
    # Get response
    try:
        response = llm_agent.get_response_content(messages, reasoning_effort='low', response_format=response_format)
        print(f"Agreement: {response.agreement}")
        print(f"Disagreement areas: '{response.disagreement_areas}'")
        if hasattr(response, 'reason'):
            print(f"Reasoning: {response.reason}")
    except Exception as e:
        print(f"Error: {e}")

# Test all fiscal agreement prompts with sample data
for prompt_file in sorted(prompts['fiscal_agreement']):
    for sample_name, sample_data in fiscal_agreement_samples.items():
        test_fiscal_agreement(prompt_file, sample_data, sample_name)
        break  # Only test with first sample for brevity


=== Testing fiscal_agreement_chain_of_thought.md with tax_policy_disagreement ===
Agreement: disagreement exists
Disagreement areas: 'Tax Policy; Government Spending'
Reasoning: IMF staff advocates comprehensive tax reforms to broaden the tax base and raise revenue, while the authority prioritizes efficiency of public expenditure and reducing waste, signaling a policy preference to avoid higher tax burdens and focus on spending reforms. This represents a disagreement on Tax Policy and the stance/direction of Government Spending.

=== Testing fiscal_agreement_few_shot.md with tax_policy_disagreement ===
Agreement: disagreement exists
Disagreement areas: 'Tax Policy; Public Spending'

=== Testing fiscal_agreement_simple.md with tax_policy_disagreement ===
Agreement: disagreement exists
Disagreement areas: 'Tax Policy; Public Spending'

=== Testing fiscal_agreement_with_definitions.md with tax_policy_disagreement ===
Agreement: disagreement exists
Disagreement areas: 'Tax Policy; Governm

## 5. Comprehensive Test with All Variants

In [13]:
def run_comprehensive_test():
    """Run a comprehensive test comparing all prompt variants on the same data"""
    print("\n" + "="*80)
    print("COMPREHENSIVE COMPARISON TEST")
    print("="*80)
    
    # Test monetary stance variants on the same sample
    print("\n🔹 MONETARY STANCE VARIANTS COMPARISON")
    sample_data = monetary_samples['accommodative_tightening']
    for prompt_file in sorted(prompts['monetary_stance']):
        variant_name = os.path.basename(prompt_file).replace('monetary_stance_', '').replace('.md', '')
        print(f"\n--- {variant_name.upper()} ---")
        test_monetary_stance(prompt_file, sample_data, 'accommodative_tightening')
    
    # Test monetary agreement variants on the same sample
    print("\n🔹 MONETARY AGREEMENT VARIANTS COMPARISON")
    sample_data = monetary_agreement_samples['disagreement']
    for prompt_file in sorted(prompts['monetary_agreement']):
        variant_name = os.path.basename(prompt_file).replace('monetary_agreement_', '').replace('.md', '')
        print(f"\n--- {variant_name.upper()} ---")
        test_monetary_agreement(prompt_file, sample_data, 'disagreement')
    
    # Test fiscal stance variants on the same sample
    print("\n🔹 FISCAL STANCE VARIANTS COMPARISON")
    sample_data = fiscal_stance_samples['consolidation_vs_expansion']
    for prompt_file in sorted(prompts['fiscal_stance']):
        variant_name = os.path.basename(prompt_file).replace('fiscal_stance_', '').replace('.md', '')
        print(f"\n--- {variant_name.upper()} ---")
        test_fiscal_stance(prompt_file, sample_data, 'consolidation_vs_expansion')
    
    # Test fiscal agreement variants on the same sample
    print("\n🔹 FISCAL AGREEMENT VARIANTS COMPARISON")
    sample_data = fiscal_agreement_samples['tax_policy_disagreement']
    for prompt_file in sorted(prompts['fiscal_agreement']):
        variant_name = os.path.basename(prompt_file).replace('fiscal_agreement_', '').replace('.md', '')
        print(f"\n--- {variant_name.upper()} ---")
        test_fiscal_agreement(prompt_file, sample_data, 'tax_policy_disagreement')

# Run comprehensive test
# Uncomment the line below to run the full comparison
# run_comprehensive_test()

## 6. Quick Demo Test

In [14]:
# Quick demo with one example from each category
print("Quick Demo Test - One example from each category:\n")

# Monetary stance (chain of thought version)
monetary_stance_file = [f for f in prompts['monetary_stance'] if 'chain_of_thought' in f][0]
test_monetary_stance(monetary_stance_file, monetary_samples['accommodative_tightening'], 'demo')

# Monetary agreement (few shot version)
monetary_agreement_file = [f for f in prompts['monetary_agreement'] if 'few_shot' in f][0]
test_monetary_agreement(monetary_agreement_file, monetary_agreement_samples['disagreement'], 'demo')

# Fiscal stance (simple version)
fiscal_stance_file = [f for f in prompts['fiscal_stance'] if 'simple' in f][0]
test_fiscal_stance(fiscal_stance_file, fiscal_stance_samples['consolidation_vs_expansion'], 'demo')

# Fiscal agreement (with definitions version)
fiscal_agreement_file = [f for f in prompts['fiscal_agreement'] if 'with_definitions' in f][0]
test_fiscal_agreement(fiscal_agreement_file, fiscal_agreement_samples['tax_policy_disagreement'], 'demo')

Quick Demo Test - One example from each category:


=== Testing monetary_stance_chain_of_thought.md with demo ===
Current stance: accommodative
Future direction: tightening
Reasoning: The text explicitly states the current policy is accommodative and should gradually move towards a neutral stance in 2024, which implies a tightening relative to the accommodative position. The discussion of pass-through and monitoring does not change the directional assessment beyond this.

=== Testing monetary_agreement_few_shot.md with demo ===
Agreement: disagreement exists
Disagreement areas: 'Future Policy Direction; Monetary Policy Stance'

=== Testing fiscal_stance_simple.md with demo ===
Staff current: moderately contractionary
Staff future: contractionary
Authority current: expansionary
Authority future: expansionary
Agreement other: irrelevant

=== Testing fiscal_agreement_with_definitions.md with demo ===
Agreement: disagreement exists
Disagreement areas: 'Tax Policy; Government Spending'


## Summary

This notebook demonstrates all 16 fiscal and monetary policy classification prompts:

### Monetary Policy (8 prompts):
- **Stance Classification** (4): simple, with_definitions, chain_of_thought, few_shot
- **Agreement Classification** (4): simple, with_definitions, chain_of_thought, few_shot

### Fiscal Policy (8 prompts):
- **Stance Classification** (4): simple, with_definitions, chain_of_thought, few_shot
- **Agreement Classification** (4): simple, with_definitions, chain_of_thought, few_shot

### Key Features Tested:
- ✅ **Template Loading**: All prompts load from markdown files
- ✅ **Structured Outputs**: Pydantic models ensure type safety
- ✅ **Variable Substitution**: Country, year, text placeholders
- ✅ **Multiple Variants**: Different prompt engineering approaches
- ✅ **Real Examples**: Realistic IMF-style text samples

Use the `run_comprehensive_test()` function to compare all variants on the same data.